# Filter Code Study — Figures Notebook

This notebook generates **publication-ready PDF figures** for:
- Token-level metrics by *model family × variant* (e.g., CodeBLEU, ROUGE-L)
- Semantics-aware metrics by *(model, variant)* (skip/effects/path/platform alignment)
- SUS vs NASA-TLX scatter plot

## Inputs you need
1. A dataframe `df` with columns at least:
   - `model`, `variant`
   - token-level metrics: `codebleu`, `codebert_f1`, `bleu`, `meteor`, `rouge1`, `rouge2`, `rougeL`
2. (Optional) A dataframe `df_metrics` with columns at least:
   - `model`, `variant`, and semantic/platform metrics such as:
     `skip_similarity`, `skip_target_similarity`, `effect_similarity`, `semantic_similarity`, `path_similarity`,
     `api_coverage_score`, `api_precision_score`, `api_usage_score`, `platform_alignment`

> If you already have them in memory, just skip the loading cells.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Output folders (PDFs will be written here) ----
OUT_DIR = "figures"
OUT_FAMILY = os.path.join(OUT_DIR, "metrics_by_family")
OUT_SEM = os.path.join(OUT_DIR, "semantic_metrics")
OUT_USER = os.path.join(OUT_DIR, "user_study")

for d in [OUT_DIR, OUT_FAMILY, OUT_SEM, OUT_USER]:
    os.makedirs(d, exist_ok=True)

OUT_DIR, OUT_FAMILY, OUT_SEM, OUT_USER


## 1) Load your results dataframe(s)

### Option A — load from CSV
- Put your main results CSV at `./results.csv` (or change the path).
- Put your semantic metrics CSV at `./df_metrics.csv` (optional).

### Option B — if you already have `df` / `df_metrics` in memory
Just run the next cells that build plots.


In [ ]:
# --- Option A: load df from CSV (edit paths as needed) ---
RESULTS_CSV = "results.csv"   # <-- change if needed
DF_METRICS_CSV = "df_metrics.csv"  # optional

df = None
df_metrics = None

if os.path.exists(RESULTS_CSV):
    df = pd.read_csv(RESULTS_CSV)
    print("Loaded df:", df.shape)
else:
    print("results.csv not found — if you already have df in memory, ignore this.")

if os.path.exists(DF_METRICS_CSV):
    df_metrics = pd.read_csv(DF_METRICS_CSV)
    print("Loaded df_metrics:", df_metrics.shape)
else:
    print("df_metrics.csv not found — semantic plots will be skipped unless df_metrics is provided.")


## 2) Token-level plots by model family × variant

Outputs are saved as PDF files in `figures/metrics_by_family/`.


In [ ]:
def get_family(name: str) -> str:
    name_l = str(name).lower()
    if "codellama" in name_l:
        return "codellama"
    if "codestral" in name_l:
        return "codestral"
    if "deepseek" in name_l:
        return "deepseek"
    if "qwen" in name_l:
        return "qwen"
    if "mistral" in name_l:
        return "mistral"
    if "llama" in name_l:
        return "llama"
    return str(name)

def build_family_means(df: pd.DataFrame) -> pd.DataFrame:
    metric_cols = ['codebleu','codebert_f1','bleu','meteor','rouge1','rouge2','rougeL']
    needed = {'model','variant'} | set(metric_cols)
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"df is missing columns: {missing}")
    out = (
        df.copy()
          .groupby(['model','variant'])[metric_cols]
          .mean()
          .reset_index()
    )
    out['family'] = out['model'].apply(get_family)
    return out

def plot_metric_by_family_variant(df_plot: pd.DataFrame, metric: str, out_dir: str) -> str:
    families = list(df_plot['family'].unique())
    variants = list(df_plot['variant'].unique())

    x = np.arange(len(families))
    width = 0.8 / max(1, len(variants))

    plt.figure(figsize=(8, 4))
    for i, v in enumerate(variants):
        vals = []
        for fam in families:
            row = df_plot[(df_plot['family'] == fam) & (df_plot['variant'] == v)]
            vals.append(float(row[metric].values[0]) if len(row) > 0 else 0.0)
        plt.bar(x + i * width, vals, width, label=v)

    plt.xticks(x + width * (len(variants)-1) / 2, families)
    plt.ylabel(metric)
    plt.title(f"{metric} by model family (variants)")
    plt.legend()
    plt.tight_layout()

    out_path = os.path.join(out_dir, f"{metric}_family_variant.pdf")
    plt.savefig(out_path)
    plt.close()
    return out_path

if df is None:
    raise RuntimeError("df is not loaded. Load it or assign df before plotting.")

df_plot = build_family_means(df)

saved_files = []
for metric in ["codebleu", "rougeL", "codebert_f1"]:
    saved_files.append(plot_metric_by_family_variant(df_plot, metric, OUT_FAMILY))

saved_files


## 3) Semantics-aware plots by (model, variant)

Outputs are saved as PDF files in `figures/semantic_metrics/`.


In [ ]:
def plot_results_by_model_variant(df_metrics: pd.DataFrame, metric_col: str, out_dir: str,
                                  exclude_parse_errors: bool = True) -> str:
    dfm = df_metrics.copy()
    if exclude_parse_errors and 'status' in dfm.columns:
        dfm = dfm[dfm['status'] != 'parse_error']

    for col in ['model','variant',metric_col]:
        if col not in dfm.columns:
            raise ValueError(f"df_metrics missing column: {col}")

    grouped = (
        dfm.groupby(['model','variant'])[metric_col]
           .mean()
           .dropna()
           .sort_values()
    )
    if grouped.empty:
        raise ValueError(f"No data available for metric: {metric_col}")

    labels = [f"{m}\n{v}" for (m,v) in grouped.index]
    values = grouped.values

    plt.figure(figsize=(max(6, len(labels)*0.7), 4))
    plt.bar(range(len(values)), values)
    plt.xticks(range(len(values)), labels, rotation=45, ha='right')
    plt.ylabel(metric_col)
    plt.title(metric_col)
    plt.tight_layout()

    out_path = os.path.join(out_dir, f"{metric_col}.pdf")
    plt.savefig(out_path)
    plt.close()
    return out_path

semantic_files = []
if df_metrics is not None:
    for metric_col in [
        'skip_similarity','skip_target_similarity','effect_similarity',
        'semantic_similarity','path_similarity',
        'api_coverage_score','api_precision_score','api_usage_score',
        'platform_alignment'
    ]:
        if metric_col in df_metrics.columns:
            semantic_files.append(plot_results_by_model_variant(df_metrics, metric_col, OUT_SEM))
        else:
            print(f"Skipping {metric_col}: column not found")
else:
    print("df_metrics not provided; skipping semantic plots.")

semantic_files


## 4) SUS vs NASA-TLX scatter plot

Place the questionnaire CSV next to the notebook (or set the path below).
Output is saved in `figures/user_study/sus_tlx_scatter.pdf`.


In [ ]:
QUESTIONNAIRE_CSV = "Filter Code User Study – SUS + NASA-TLX.csv"  # change if needed

if not os.path.exists(QUESTIONNAIRE_CSV):
    raise FileNotFoundError(f"Questionnaire CSV not found: {QUESTIONNAIRE_CSV}")

qdf = pd.read_csv(QUESTIONNAIRE_CSV)

sus_cols = [c for c in qdf.columns if c.strip().startswith(tuple(str(i) for i in range(1,11)))]
sus = qdf[sus_cols].apply(pd.to_numeric, errors='coerce')
odd_idx = [0,2,4,6,8]
even_idx = [1,3,5,7,9]
sus_score = ((sus.iloc[:,odd_idx]-1).sum(axis=1) + (5-sus.iloc[:,even_idx]).sum(axis=1))*2.5

tlx_cols = ['Domanda Mentale','Domanda Fisica','Domanda Temporale','Prestazione','Sforzo','Frustrazione']
tlx = qdf[tlx_cols].apply(pd.to_numeric, errors='coerce')
tlx_raw = tlx.mean(axis=1)

plt.figure()
plt.scatter(sus_score, tlx_raw)
plt.xlabel("SUS (0–100)")
plt.ylabel("NASA-TLX Raw (0–10)")
plt.title("SUS vs NASA-TLX (Raw)")

m, b = np.polyfit(sus_score, tlx_raw, 1)
x = np.linspace(sus_score.min(), sus_score.max(), 100)
plt.plot(x, m*x + b)

plt.tight_layout()
out_path = os.path.join(OUT_USER, "sus_tlx_scatter.pdf")
plt.savefig(out_path)
plt.close()
print("Saved:", out_path)


## Done

All PDFs are saved under `figures/`:
- `figures/metrics_by_family/`
- `figures/semantic_metrics/`
- `figures/user_study/`
